In [ ]:
!pip install "huggingface-hub==0.35.3"
!pip install "transformers==4.57.0"
!pip install triton
!pip install xxhash
!pip install "jax[tpu]==0.4.35" \
  -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

Looking in links: https://storage.googleapis.com/jax-releases/libtpu_releases.html


In [ ]:
!git clone https://github.com/Kaminyou/nano-vLLM-TPU.git
!cd nano-vLLM-TPU && git checkout temp/workable-tpu
from huggingface_hub import snapshot_download
!mkdir Qwen3-0.6B
snapshot_download(
    repo_id="Qwen/Qwen3-0.6B",
    local_dir="./Qwen3-0.6B",
    local_dir_use_symlinks=False,
    resume_download=True
)

fatal: destination path 'nano-vLLM-TPU' already exists and is not an empty directory.
Already on 'temp/workable-tpu'
Your branch is up to date with 'origin/temp/workable-tpu'.
mkdir: cannot create directory ‘Qwen3-0.6B’: File exists


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'/content/Qwen3-0.6B'

In [ ]:
%%writefile nano_vllm_v4.py
# -*- coding: utf-8 -*-
import os
import jax
import jax.numpy as jnp

from dataclasses import dataclass, field
from typing import Optional
import numpy as np

from glob import glob
import numpy as np
from safetensors import safe_open
import functools
from typing import NamedTuple
from transformers import Qwen3Config

from copy import copy
from enum import Enum, auto
from itertools import count
from dataclasses import dataclass
from collections import deque
import xxhash
from transformers import AutoConfig

import atexit
from dataclasses import fields
from time import perf_counter
from tqdm.auto import tqdm
from transformers import AutoTokenizer

@dataclass
class Context:
    is_prefill: bool = False
    # prefill
    cu_seqlens_q: Optional[np.ndarray] = None       # [batch+1]  int32
    slot_mapping: Optional[np.ndarray] = None       # [total_tokens] int32
    padding_mask: Optional[np.ndarray] = None       # [batch, padded_len] bool
    batch_size: int = 0
    padded_seq_len: int = 0
    # decode
    context_lens: Optional[np.ndarray] = None       # [batch] int32
    block_tables: Optional[np.ndarray] = None       # [batch, max_blocks] int32

_CONTEXT = Context()

def get_context() -> Context:
    return _CONTEXT

def set_context(is_prefill, *, cu_seqlens_q=None, slot_mapping=None,
                padding_mask=None, batch_size=0, padded_seq_len=0,
                context_lens=None, block_tables=None):
    global _CONTEXT
    _CONTEXT = Context(
        is_prefill=is_prefill,
        cu_seqlens_q=cu_seqlens_q,
        slot_mapping=slot_mapping,
        padding_mask=padding_mask,
        batch_size=batch_size,
        padded_seq_len=padded_seq_len,
        context_lens=context_lens,
        block_tables=block_tables,
    )

def reset_context():
    global _CONTEXT
    _CONTEXT = Context()

def load_weights_from_safetensors(path: str) -> dict:
    """Returns a flat dict  {weight_name: np.ndarray}."""
    weights = {}
    for file in glob(os.path.join(path, "*.safetensors")):
        with safe_open(file, framework="numpy") as f:
            for key in f.keys():
                weights[key] = f.get_tensor(key)
    return weights


# ── Helper ────────────────────────────────────────────────────────────────
def silu_and_mul(x: jnp.ndarray) -> jnp.ndarray:
    """SiGLU: silu(gate) * up"""
    gate, up = jnp.split(x, 2, axis=-1)
    return jax.nn.silu(gate) * up


def rms_norm(x: jnp.ndarray, weight: jnp.ndarray, eps: float = 1e-6) -> jnp.ndarray:
    """Root-mean-square layer normalisation."""
    variance = jnp.mean(jnp.square(x.astype(jnp.float32)), axis=-1, keepdims=True)
    x_normed = x * jax.lax.rsqrt(variance + eps)
    return (weight * x_normed).astype(x.dtype)


def rms_norm_residual(
    x: jnp.ndarray,
    residual: jnp.ndarray,
    weight: jnp.ndarray,
    eps: float = 1e-6,
) -> tuple:
    """Fused residual add + RMS-norm (saves a read of x)."""
    x = x + residual
    return rms_norm(x, weight, eps), x


# ── Sharding helpers (single-device stub, tp_size=1) ─────────────────────
def linear(x: jnp.ndarray, w: jnp.ndarray, bias=None) -> jnp.ndarray:
    y = x @ w.T
    if bias is not None:
        y = y + bias
    return y


# ── RoPE ──────────────────────────────────────────────────────────────────
def precompute_rope_cache(
    head_dim: int,
    max_positions: int,
    base: float = 10000.0,
    dtype=jnp.bfloat16,
) -> jnp.ndarray:
    """Returns cache of shape [max_positions, 1, head_dim] with (cos, sin) packed."""
    inv_freq = 1.0 / (base ** (np.arange(0, head_dim, 2, dtype=np.float32) / head_dim))
    t = np.arange(max_positions, dtype=np.float32)
    freqs = np.outer(t, inv_freq)          # [T, head_dim/2]
    cos = np.cos(freqs).astype(np.float32)
    sin = np.sin(freqs).astype(np.float32)
    cache = np.concatenate([cos, sin], axis=-1)[:, None, :]  # [T, 1, head_dim]
    return jnp.array(cache, dtype=jnp.float32)  # kept fp32 for accuracy


def apply_rope(
    x: jnp.ndarray,          # [seq, heads, head_dim]
    cos: jnp.ndarray,         # [seq, 1, head_dim/2]
    sin: jnp.ndarray,         # [seq, 1, head_dim/2]
) -> jnp.ndarray:
    x1, x2 = jnp.split(x.astype(jnp.float32), 2, axis=-1)
    y = jnp.concatenate([x1 * cos - x2 * sin, x2 * cos + x1 * sin], axis=-1)
    return y.astype(x.dtype)


# -- Import SplashAttention (the GQA-compatible TPU flash kernel) ----------
try:
    from jax.experimental.pallas.ops.tpu.splash_attention import (
        splash_attention_kernel,
        splash_attention_mask,
    )
    _HAS_SPLASH = True
    print('SplashAttention (GQA-capable TPU Pallas kernel) loaded')
except ImportError:
    _HAS_SPLASH = False
    print('SplashAttention not found - falling back to XLA SDPA')

# Keep the old flash_attention flag name for backward compat with cell 8
_HAS_PALLAS_FLASH = _HAS_SPLASH



# -- KV-cache scatter (store tokens into paged cache) ----------------------
def store_kvcache(
    key: jnp.ndarray,         # [num_tokens, num_kv_heads, head_dim]
    value: jnp.ndarray,
    k_cache: jnp.ndarray,     # [num_blocks, block_size, num_kv_heads, head_dim]
    v_cache: jnp.ndarray,
    slot_mapping: np.ndarray, # [num_tokens] int32  (host numpy)
) -> tuple:
    num_blocks, block_size, num_kv_heads, head_dim = k_cache.shape
    k_flat = k_cache.reshape(-1, num_kv_heads, head_dim)
    v_flat = v_cache.reshape(-1, num_kv_heads, head_dim)
    slot_idx = jnp.array(slot_mapping, dtype=jnp.int32)
    k_flat = k_flat.at[slot_idx].set(key)
    v_flat = v_flat.at[slot_idx].set(value)
    return (k_flat.reshape(num_blocks, block_size, num_kv_heads, head_dim),
            v_flat.reshape(num_blocks, block_size, num_kv_heads, head_dim))


# -- SplashAttention prefill -----------------------------------------------
@functools.lru_cache(maxsize=32)
def _make_splash_kernel(num_q_heads: int, seq_len: int):
    """
    Build (and cache) a SplashAttention kernel for a given (num_q_heads, seq_len).

    SplashAttention supports GQA
    """
    mask_shape = (seq_len, seq_len)
    # CausalMask encodes the lower-triangular causal attention pattern
    causal = splash_attention_mask.CausalMask(shape=mask_shape)
    # MultiHeadMask: one mask entry per *query* head
    multi  = splash_attention_mask.MultiHeadMask(
        masks=[causal] * num_q_heads
    )
    # make_splash_mha returns a callable that takes (q, k, v) with shapes
    # [num_q_heads, seq_len, head_dim] and [num_kv_heads, seq_len, head_dim]
    # head_shards=1, q_seq_shards=1 for single-device usage
    kernel = splash_attention_kernel.make_splash_mha(
        mask=multi,
        head_shards=1,
        q_seq_shards=1,
    )
    return kernel


def _prefill_attention_splash(
    q: jnp.ndarray,           # [batch, num_q_heads,  seq, head_dim]
    k: jnp.ndarray,           # [batch, num_kv_heads, seq, head_dim]
    v: jnp.ndarray,           # [batch, num_kv_heads, seq, head_dim]
    padding_mask: jnp.ndarray,# [batch, seq]  bool
    scale: float,
) -> jnp.ndarray:             # [batch, num_q_heads, seq, head_dim]
    batch, num_q_heads, seq, head_dim = q.shape
    kernel = _make_splash_kernel(num_q_heads, seq)

    # Build segment IDs: real tokens -> unique position (1..S), pads -> 0
    # Shape [B, S] int32
    positions = jnp.arange(1, seq + 1, dtype=jnp.int32)[None, :]  # [1, S]
    #seg = jnp.where(padding_mask, positions, 0).astype(jnp.int32)
    seg = jnp.where(padding_mask, 1, 0).astype(jnp.int32)  # [B, S]  pads=0, real=1 (all real tokens in same segment)

    # segment_ids for SplashAttention: SegmentIds(q=[B,S], kv=[B,S])
    # We pass it per-batch-element via vmap
    try:
        from jax.experimental.pallas.ops.tpu.splash_attention.splash_attention_kernel import SegmentIds as SplashSegmentIds
        seg_ids = SplashSegmentIds(q=seg, kv=seg)
        # vmap over batch dim; kernel signature: (q, k, v, segment_ids)
        vmapped = jax.vmap(lambda qi, ki, vi, si: kernel(qi, ki, vi, segment_ids=si))
        out = vmapped(q, k, v, seg_ids)
    except Exception:
        # Fallback: no segment_ids, just causal mask
        vmapped = jax.vmap(kernel)
        out = vmapped(q, k, v)

    # pre-scale q before the call so effective scale = scale
    return out   # [B, Hq, S, D]


def _prefill_attention_splash_scaled(
    q: jnp.ndarray,
    k: jnp.ndarray,
    v: jnp.ndarray,
    padding_mask: jnp.ndarray,
    scale: float,
) -> jnp.ndarray:
    """Wrapper that pre-scales q by `scale` before calling SplashAttention."""
    return _prefill_attention_splash(q * scale, k, v, padding_mask, scale)


# -- XLA SDPA fallback (used when SplashAttention unavailable) -------------
def _prefill_attention_xla(
    q: jnp.ndarray,           # [B, Hq, S, D]
    k: jnp.ndarray,           # [B, Hkv, S, D]  (already expanded if needed)
    v: jnp.ndarray,
    padding_mask: jnp.ndarray,# [B, S]
    scale: float,
) -> jnp.ndarray:
    """Pure XLA scaled dot-product attention (MHA, GQA with pre-expanded k/v)."""
    B, Hq, S, D = q.shape
    # Causal + padding mask
    causal   = jnp.tril(jnp.ones((S, S), dtype=jnp.bool_))
    pad      = padding_mask[:, None, None, :]              # [B,1,1,S]
    mask     = causal[None, None, :, :] & pad              # [B,1,S,S]
    attn_bias = jnp.where(mask, 0.0, jnp.finfo(q.dtype).min)
    scores = jnp.einsum('bhsd,bhkd->bhsk', q, k) * scale + attn_bias
    probs  = jax.nn.softmax(scores.astype(jnp.float32), axis=-1).astype(q.dtype)
    return jnp.einsum('bhsk,bhkd->bhsd', probs, v)


# -- Decode attention: PagedAttention gather --------------------------------
def _decode_attention(
    q: jnp.ndarray,           # [batch, num_q_heads, 1, head_dim]
    k_cache: jnp.ndarray,     # [num_blocks, block_size, num_kv_heads, head_dim]
    v_cache: jnp.ndarray,
    block_tables: np.ndarray, # [batch, max_blocks]  int32  host
    context_lens: np.ndarray, # [batch]              int32  host
    scale: float,
    num_q_per_kv: int,
) -> jnp.ndarray:             # [batch, num_q_heads, head_dim]
    num_blocks_per_seq = block_tables.shape[1]
    block_size   = k_cache.shape[1]
    num_kv_heads = k_cache.shape[2]
    head_dim     = k_cache.shape[3]
    batch_size   = q.shape[0]
    num_q_heads  = q.shape[1]
    total_ctx    = num_blocks_per_seq * block_size

    # Gather all blocks for each sequence
    bt = jnp.maximum(jnp.asarray(block_tables, dtype=jnp.int32), 0)  # [B, max_blocks]
    flat_bt  = bt.reshape(-1)                                               # [B*max_blocks]
    k_g = k_cache[flat_bt].reshape(batch_size, total_ctx, num_kv_heads, head_dim)
    v_g = v_cache[flat_bt].reshape(batch_size, total_ctx, num_kv_heads, head_dim)

    k_t = k_g.transpose(0, 2, 1, 3)   # [B, Hkv, total_ctx, D]
    v_t = v_g.transpose(0, 2, 1, 3)

    # GQA expansion (repeat kv heads to match q heads)
    # if num_q_per_kv > 1:
    #    k_t = jnp.repeat(k_t, num_q_per_kv, axis=1)  # [B, Hq, total_ctx, D]
    #    v_t = jnp.repeat(v_t, num_q_per_kv, axis=1)

    # Causal mask: attend only up to context_len
    #seq_idx  = jnp.arange(total_ctx, dtype=jnp.int32)[None, :]  # [1, total_ctx]
    #ctx_len  = jnp.asarray(context_lens, dtype=jnp.int32)[:, None] # [B, 1]
    #valid    = seq_idx < ctx_len                                  # [B, total_ctx]
    #attn_bias = jnp.where(valid[:, None, None, :], 0.0,
     #                     jnp.finfo(q.dtype).min)                 # [B,1,1,total_ctx]

    #scores = jnp.einsum('bh1d,bhkd->bh1k', q, k_t) * scale + attn_bias
    #probs  = jax.nn.softmax(scores.astype(jnp.float32), axis=-1).astype(q.dtype)
    #out    = jnp.einsum('bh1k,bhkd->bh1d', probs, v_t)
    #return out.squeeze(2)   # [B, Hq, D]
    seq_idx  = jnp.arange(total_ctx, dtype=jnp.int32)[None, :]
    ctx_len  = jnp.asarray(context_lens, dtype=jnp.int32)[:, None]
    valid    = seq_idx < ctx_len

    if num_q_per_kv > 1:
        # GQA without materialising repeated K/V copies.
        # Reshape q from [B, Hq, 1, D] → [B, Hkv, G, 1, D] where G = num_q_per_kv
        q_r = q.reshape(batch_size, num_kv_heads, num_q_per_kv, 1, head_dim)
        attn_bias = jnp.where(valid[:, None, None, None, :], 0.0,
                              jnp.finfo(q.dtype).min)             # [B,1,1,1,total_ctx]
        scores = jnp.einsum('bhgqd,bhtd->bhgqt', q_r, k_t) * scale + attn_bias
        probs  = jax.nn.softmax(scores.astype(jnp.float32), axis=-1).astype(q.dtype)
        out    = jnp.einsum('bhgqt,bhtd->bhgqd', probs, v_t)     # [B, Hkv, G, 1, D]
        return out.reshape(batch_size, num_q_heads, head_dim)      # [B, Hq, D]
    else:
        attn_bias = jnp.where(valid[:, None, None, :], 0.0,
                              jnp.finfo(q.dtype).min)             # [B,1,1,total_ctx]
        scores = jnp.einsum('bh1d,bhkd->bh1k', q, k_t) * scale + attn_bias
        probs  = jax.nn.softmax(scores.astype(jnp.float32), axis=-1).astype(q.dtype)
        out    = jnp.einsum('bh1k,bhkd->bh1d', probs, v_t)
        return out.squeeze(2)



@jax.tree_util.register_pytree_node_class
class Qwen3Weights:
    """
    JAX-friendly weight container.

    - shared: non-layer weights
    - layers: per-layer weights stacked on axis 0
    - cfg: kept as static aux data in the pytree
    """

    def __init__(self, path: str, hf_config):
        self.cfg = hf_config
        raw = load_weights_from_safetensors(path)

        head_dim = getattr(
            hf_config, "head_dim", hf_config.hidden_size // hf_config.num_attention_heads
        )
        rope_cache = precompute_rope_cache(
            head_dim,
            max_positions=hf_config.max_position_embeddings,
            base=float(getattr(hf_config, "rope_theta", 1_000_000)),
        )

        self.shared = {
            "model.embed_tokens.weight": jnp.asarray(
                raw.pop("model.embed_tokens.weight"), dtype=jnp.bfloat16
            ),
            "model.norm.weight": jnp.asarray(
                raw.pop("model.norm.weight"), dtype=jnp.bfloat16
            ),
        }

        lm_head = raw.pop("lm_head.weight", None)
        if lm_head is not None:
            self.shared["lm_head.weight"] = jnp.asarray(lm_head, dtype=jnp.bfloat16)

        layers = {
            "input_layernorm.weight": [],
            "post_attention_layernorm.weight": [],
            "self_attn.qkv_proj.weight": [],
            "self_attn.o_proj.weight": [],
            "self_attn.q_norm.weight": [],
            "self_attn.k_norm.weight": [],
            "mlp.gate_up_proj.weight": [],
            "mlp.down_proj.weight": [],
        }
        has_qkv_bias = f"model.layers.0.self_attn.q_proj.bias" in raw
        qkv_biases = []

        for i in range(hf_config.num_hidden_layers):
            pfx = f"model.layers.{i}"
            attn = f"{pfx}.self_attn"
            mlp = f"{pfx}.mlp"

            q = raw.pop(f"{attn}.q_proj.weight")
            k = raw.pop(f"{attn}.k_proj.weight")
            v = raw.pop(f"{attn}.v_proj.weight")
            layers["self_attn.qkv_proj.weight"].append(np.concatenate([q, k, v], axis=0))

            if has_qkv_bias:
                qkv_biases.append(
                    np.concatenate(
                        [
                            raw.pop(f"{attn}.q_proj.bias"),
                            raw.pop(f"{attn}.k_proj.bias"),
                            raw.pop(f"{attn}.v_proj.bias"),
                        ],
                        axis=0,
                    )
                )

            layers["input_layernorm.weight"].append(
                raw.pop(f"{pfx}.input_layernorm.weight")
            )
            layers["post_attention_layernorm.weight"].append(
                raw.pop(f"{pfx}.post_attention_layernorm.weight")
            )
            layers["self_attn.o_proj.weight"].append(raw.pop(f"{attn}.o_proj.weight"))
            layers["self_attn.q_norm.weight"].append(raw.pop(f"{attn}.q_norm.weight"))
            layers["self_attn.k_norm.weight"].append(raw.pop(f"{attn}.k_norm.weight"))
            layers["mlp.gate_up_proj.weight"].append(
                np.concatenate(
                    [
                        raw.pop(f"{mlp}.gate_proj.weight"),
                        raw.pop(f"{mlp}.up_proj.weight"),
                    ],
                    axis=0,
                )
            )
            layers["mlp.down_proj.weight"].append(raw.pop(f"{mlp}.down_proj.weight"))

        self.shared["rope_cache"] = jnp.asarray(rope_cache, dtype=jnp.float32)
        self.layers = {
            name: jnp.asarray(np.stack(vals, axis=0), dtype=jnp.bfloat16)
            for name, vals in layers.items()
        }
        self.layers["self_attn.qkv_proj.bias"] = (
            jnp.asarray(np.stack(qkv_biases, axis=0), dtype=jnp.bfloat16)
            if has_qkv_bias
            else None
        )
        self.meta = {
            "num_hidden_layers": hf_config.num_hidden_layers,
            "hidden_size": hf_config.hidden_size,
            "num_attention_heads": hf_config.num_attention_heads,
            "num_key_value_heads": hf_config.num_key_value_heads,
            "head_dim": head_dim,
            "rms_norm_eps": hf_config.rms_norm_eps,
            "attention_bias": bool(getattr(hf_config, "attention_bias", False)),
        }

    def tree_flatten(self):
        children = (self.shared, self.layers)
        aux = (self.cfg, self.meta)
        return children, aux

    @classmethod
    def tree_unflatten(cls, aux, children):
        cfg, meta = aux
        shared, layers = children
        obj = cls.__new__(cls)
        obj.cfg = cfg
        obj.meta = meta
        obj.shared = shared
        obj.layers = layers
        return obj

    def _get(self, name: str) -> jnp.ndarray:
        if name in self.shared:
            return self.shared[name]
        if name in self.layers:
            return self.layers[name]
        raise KeyError(f"Weight '{name}' not found.")

    def _get_opt(self, name: str):
        if name in self.shared:
            return self.shared[name]
        return self.layers.get(name, None)


@jax.tree_util.register_pytree_node_class
class KVCacheLayer:
    __slots__ = ("k", "v")

    def __init__(self, num_blocks, block_size, num_kv_heads, head_dim, dtype=jnp.bfloat16):
        self.k = jnp.zeros((num_blocks, block_size, num_kv_heads, head_dim), dtype=dtype)
        self.v = jnp.zeros((num_blocks, block_size, num_kv_heads, head_dim), dtype=dtype)

    def tree_flatten(self):
        return ((self.k, self.v), None)

    @classmethod
    def tree_unflatten(cls, aux, children):
        obj = cls.__new__(cls)
        obj.k, obj.v = children
        return obj


class PrefillBatch(NamedTuple):
    input_ids: jnp.ndarray       # [B, S]
    positions: jnp.ndarray       # [B, S]
    padding_mask: jnp.ndarray    # [B, S]
    slot_mapping: jnp.ndarray    # [B, S]
    cu_seqlens_q: jnp.ndarray    # [B + 1]


class DecodeBatch(NamedTuple):
    input_ids: jnp.ndarray       # [B]
    positions: jnp.ndarray       # [B]
    slot_mapping: jnp.ndarray    # [B]
    context_lens: jnp.ndarray    # [B]
    block_tables: jnp.ndarray    # [B, max_blocks]


def _store_kvcache_masked(
    key: jnp.ndarray,         # [T, Hkv, D]
    value: jnp.ndarray,
    k_cache: jnp.ndarray,     # [num_blocks, block_size, Hkv, D]
    v_cache: jnp.ndarray,
    slot_mapping: jnp.ndarray,# [T]
    valid_mask: jnp.ndarray,  # [T]
) -> tuple[jnp.ndarray, jnp.ndarray]:
    num_blocks, block_size, num_kv_heads, head_dim = k_cache.shape
    k_flat = k_cache.reshape(-1, num_kv_heads, head_dim)
    v_flat = v_cache.reshape(-1, num_kv_heads, head_dim)

    slot_idx = slot_mapping.reshape(-1).astype(jnp.int32)
    valid = valid_mask.reshape(-1).astype(jnp.bool_)

    prev_k = k_flat[slot_idx]
    prev_v = v_flat[slot_idx]
    key_to_write = jnp.where(valid[:, None, None], key, prev_k)
    val_to_write = jnp.where(valid[:, None, None], value, prev_v)

    k_flat = k_flat.at[slot_idx].set(key_to_write)
    v_flat = v_flat.at[slot_idx].set(val_to_write)
    return (
        k_flat.reshape(num_blocks, block_size, num_kv_heads, head_dim),
        v_flat.reshape(num_blocks, block_size, num_kv_heads, head_dim),
    )


def qwen3_attention_forward(
    hidden: jnp.ndarray,       # [T, hidden]
    positions: jnp.ndarray,    # [T]
    kv_layer: KVCacheLayer,
    weights: Qwen3Weights,
    layer_params: dict,
    batch,
    is_prefill: bool,
) -> tuple[jnp.ndarray, KVCacheLayer]:
    meta = weights.meta
    h = meta["hidden_size"]
    num_heads = meta["num_attention_heads"]
    num_kv_heads = meta["num_key_value_heads"]
    head_dim = meta["head_dim"]
    q_size = num_heads * head_dim
    kv_size = num_kv_heads * head_dim
    scale = head_dim ** -0.5
    num_q_per_kv = num_heads // num_kv_heads

    T = hidden.shape[0]
    qkv = linear(
        hidden,
        layer_params["qkv_w"],
        layer_params["qkv_b"] if meta["attention_bias"] else None,
    )
    q, k, v = jnp.split(qkv, [q_size, q_size + kv_size], axis=-1)

    q = q.reshape(T, num_heads, head_dim)
    k = k.reshape(T, num_kv_heads, head_dim)
    v = v.reshape(T, num_kv_heads, head_dim)

    if not meta["attention_bias"]:
        q = rms_norm(q, layer_params["q_norm_w"], meta["rms_norm_eps"])
        k = rms_norm(k, layer_params["k_norm_w"], meta["rms_norm_eps"])

    cos_sin = weights.shared["rope_cache"][positions.astype(jnp.int32)]
    cos, sin = jnp.split(cos_sin, 2, axis=-1)
    q = apply_rope(q, cos, sin)
    k = apply_rope(k, cos, sin)

    if is_prefill:
        flat_mask = batch.padding_mask.reshape(-1)
        new_k, new_v = _store_kvcache_masked(
            k,
            v,
            kv_layer.k,
            kv_layer.v,
            batch.slot_mapping.reshape(-1),
            flat_mask,
        )
    else:
        new_k, new_v = store_kvcache(
            k,
            v,
            kv_layer.k,
            kv_layer.v,
            batch.slot_mapping,
        )
    new_kv = KVCacheLayer.__new__(KVCacheLayer)
    new_kv.k = new_k
    new_kv.v = new_v

    if is_prefill:
        B, S = batch.input_ids.shape
        q_b = q.reshape(B, S, num_heads, head_dim).transpose(0, 2, 1, 3)
        k_b = k.reshape(B, S, num_kv_heads, head_dim).transpose(0, 2, 1, 3)
        v_b = v.reshape(B, S, num_kv_heads, head_dim).transpose(0, 2, 1, 3)
        if _HAS_PALLAS_FLASH:
            o_b = _prefill_attention_splash_scaled(q_b, k_b, v_b, batch.padding_mask, scale)
        else:
            if num_q_per_kv > 1:
                k_b = jnp.repeat(k_b, num_q_per_kv, axis=1)
                v_b = jnp.repeat(v_b, num_q_per_kv, axis=1)
            o_b = _prefill_attention_xla(q_b, k_b, v_b, batch.padding_mask, scale)
        attn_out = o_b.transpose(0, 2, 1, 3).reshape(B * S, num_heads * head_dim)
    else:
        B = T
        q_b = q.reshape(B, num_heads, 1, head_dim)
        o_b = _decode_attention(
            q_b,
            new_kv.k,
            new_kv.v,
            batch.block_tables,
            batch.context_lens,
            scale,
            num_q_per_kv,
        )
        attn_out = o_b.reshape(B, num_heads * head_dim)

    return linear(attn_out, layer_params["o_proj_w"]), new_kv


def qwen3_mlp_forward(x: jnp.ndarray, layer_params: dict) -> jnp.ndarray:
    return linear(
        silu_and_mul(linear(x, layer_params["gate_up_w"])),
        layer_params["down_proj_w"],
    )


def qwen3_decoder_layer_forward(
    hidden: jnp.ndarray,
    residual,
    positions: jnp.ndarray,
    kv_layer: KVCacheLayer,
    weights: Qwen3Weights,
    layer_params: dict,
    batch,
    is_prefill: bool,
) -> tuple[jnp.ndarray, jnp.ndarray, KVCacheLayer]:
    eps = weights.meta["rms_norm_eps"]

    if residual is None:
        normed = rms_norm(hidden, layer_params["input_ln_w"], eps)
        residual = hidden
    else:
        normed, residual = rms_norm_residual(hidden, residual, layer_params["input_ln_w"], eps)

    attn_out, new_kv = qwen3_attention_forward(
        normed, positions, kv_layer, weights, layer_params, batch, is_prefill
    )
    hidden, residual = rms_norm_residual(attn_out, residual, layer_params["post_ln_w"], eps)
    hidden = qwen3_mlp_forward(hidden, layer_params)
    return hidden, residual, new_kv


def qwen3_forward(
    weights: Qwen3Weights,
    input_ids: jnp.ndarray,
    positions: jnp.ndarray,
    kv_caches: list[KVCacheLayer],
    batch,
    is_prefill: bool,
) -> tuple[jnp.ndarray, list[KVCacheLayer]]:
    embed_w = weights.shared["model.embed_tokens.weight"]
    hidden = embed_w[input_ids.astype(jnp.int32)]

    _qkv_b = weights.layers["self_attn.qkv_proj.bias"]
    if _qkv_b is None:
        # No bias (e.g. Qwen3); create zero dummy so lax.scan sees a concrete pytree leaf
        _qkv_b = jnp.zeros(
            weights.layers["self_attn.qkv_proj.weight"].shape[:2], dtype=jnp.bfloat16
        )
    layer_xs = {
        "input_ln_w": weights.layers["input_layernorm.weight"],
        "post_ln_w": weights.layers["post_attention_layernorm.weight"],
        "qkv_w": weights.layers["self_attn.qkv_proj.weight"],
        "qkv_b": _qkv_b,
        "o_proj_w": weights.layers["self_attn.o_proj.weight"],
        "q_norm_w": weights.layers["self_attn.q_norm.weight"],
        "k_norm_w": weights.layers["self_attn.k_norm.weight"],
        "gate_up_w": weights.layers["mlp.gate_up_proj.weight"],
        "down_proj_w": weights.layers["mlp.down_proj.weight"],
    }

    # Stack per-layer KV cache leaves so lax.scan sees a consistent leading layer axis.
    k_stack = jnp.stack([layer.k for layer in kv_caches], axis=0)
    v_stack = jnp.stack([layer.v for layer in kv_caches], axis=0)

    def scan_body(carry, xs):
        hidden, residual = carry
        layer_params, k_layer, v_layer = xs
        kv_layer = KVCacheLayer.__new__(KVCacheLayer)
        kv_layer.k = k_layer
        kv_layer.v = v_layer

        hidden, residual, new_kv = qwen3_decoder_layer_forward(
            hidden,
            residual,
            positions,
            kv_layer,
            weights,
            layer_params,
            batch,
            is_prefill,
        )
        return (hidden, residual), (new_kv.k, new_kv.v)

    residual0 = jnp.zeros_like(hidden)
    (hidden, residual), (new_k_stack, new_v_stack) = jax.lax.scan(
        scan_body,
        (hidden, residual0),
        (layer_xs, k_stack, v_stack),
    )

    hidden, _ = rms_norm_residual(
        hidden,
        residual,
        weights.shared["model.norm.weight"],
        weights.meta["rms_norm_eps"],
    )

    new_kv = []
    for i in range(new_k_stack.shape[0]):
        layer = KVCacheLayer.__new__(KVCacheLayer)
        layer.k = new_k_stack[i]
        layer.v = new_v_stack[i]
        new_kv.append(layer)
    return hidden, new_kv


def compute_logits(
    hidden: jnp.ndarray,
    weights: Qwen3Weights,
    batch,
    is_prefill: bool,
) -> jnp.ndarray:
    lm_head_w = weights.shared.get("lm_head.weight", weights.shared["model.embed_tokens.weight"])

    if is_prefill:
        last_idxs = batch.cu_seqlens_q[1:].astype(jnp.int32) - 1
        x = hidden[last_idxs]
    else:
        x = hidden
    return linear(x, lm_head_w)

def jax_sampler(logits: jnp.ndarray, temperatures: jnp.ndarray) -> jnp.ndarray:
    """
    Temperature-scaled Gumbel-max sampling.
    logits: [batch, vocab]   temperatures: [batch]
    Returns: [batch] int32 sampled token ids.
    """
    logits_f = logits.astype(jnp.float32)
    scaled   = logits_f / temperatures[:, None]
    probs    = jax.nn.softmax(scaled, axis=-1)
    # Gumbel-max trick: equivalent to multinomial sampling
    gumbel   = -jnp.log(-jnp.log(jnp.clip(probs, 1e-10, 1.0)))
    return jnp.argmax(scaled + gumbel, axis=-1).astype(jnp.int32)

# !!!IMPORTANT!!!!
# THOSE SHOULD NOT BE MODIFIED
# scheduler.py



@dataclass
class Config:
    model: str
    max_num_batched_tokens: int = 16384
    max_num_seqs: int = 512
    max_model_len: int = 4096
    gpu_memory_utilization: float = 0.9
    tensor_parallel_size: int = 1
    enforce_eager: bool = False
    hf_config: AutoConfig | None = None
    eos: int = -1
    kvcache_block_size: int = 256
    num_kvcache_blocks: int = -1

    def __post_init__(self):
        assert os.path.isdir(self.model)
        assert self.kvcache_block_size % 256 == 0
        assert 1 <= self.tensor_parallel_size <= 8
        self.hf_config = AutoConfig.from_pretrained(self.model)
        self.max_model_len = min(self.max_model_len, self.hf_config.max_position_embeddings)
        assert self.max_num_batched_tokens >= self.max_model_len

@dataclass
class SamplingParams:
    temperature: float = 1.0
    max_tokens: int = 64
    ignore_eos: bool = False

    def __post_init__(self):
        assert self.temperature > 1e-10, "greedy sampling is not permitted"

class SequenceStatus(Enum):
    WAITING = auto()
    RUNNING = auto()
    FINISHED = auto()


class Sequence:
    block_size = 256
    counter = count()

    def __init__(self, token_ids: list[int], sampling_params=SamplingParams()):
        self.seq_id = next(Sequence.counter)
        self.status = SequenceStatus.WAITING
        self.token_ids = copy(token_ids)
        self.last_token = token_ids[-1]
        self.num_tokens = len(self.token_ids)
        self.num_prompt_tokens = len(token_ids)
        self.num_cached_tokens = 0
        self.block_table = []
        self.temperature = sampling_params.temperature
        self.max_tokens = sampling_params.max_tokens
        self.ignore_eos = sampling_params.ignore_eos

    def __len__(self): return self.num_tokens
    def __getitem__(self, key): return self.token_ids[key]

    @property
    def is_finished(self): return self.status == SequenceStatus.FINISHED
    @property
    def num_completion_tokens(self): return self.num_tokens - self.num_prompt_tokens
    @property
    def prompt_token_ids(self): return self.token_ids[:self.num_prompt_tokens]
    @property
    def completion_token_ids(self): return self.token_ids[self.num_prompt_tokens:]
    @property
    def num_cached_blocks(self): return self.num_cached_tokens // self.block_size
    @property
    def num_blocks(self): return (self.num_tokens + self.block_size - 1) // self.block_size
    @property
    def last_block_num_tokens(self): return self.num_tokens - (self.num_blocks - 1) * self.block_size

    def block(self, i):
        assert 0 <= i < self.num_blocks
        return self.token_ids[i*self.block_size:(i+1)*self.block_size]

    def append_token(self, token_id: int):
        self.token_ids.append(token_id)
        self.last_token = token_id
        self.num_tokens += 1

    def __getstate__(self):
        return (self.num_tokens, self.num_prompt_tokens, self.num_cached_tokens,
                self.block_table,
                self.token_ids if self.num_completion_tokens == 0 else self.last_token)

    def __setstate__(self, state):
        self.num_tokens, self.num_prompt_tokens, self.num_cached_tokens, self.block_table = state[:-1]
        if self.num_completion_tokens == 0:
            self.token_ids = state[-1]
        else:
            self.last_token = state[-1]


class Block:
    def __init__(self, block_id):
        self.block_id = block_id; self.ref_count = 0; self.hash = -1; self.token_ids = []
    def update(self, hash, token_ids): self.hash = hash; self.token_ids = token_ids
    def reset(self): self.ref_count = 1; self.hash = -1; self.token_ids = []


class BlockManager:
    def __init__(self, num_blocks, block_size):
        self.block_size = block_size
        self.blocks = [Block(i) for i in range(num_blocks)]
        self.hash_to_block_id = {}
        self.free_block_ids: deque[int] = deque(range(num_blocks))
        self.used_block_ids: set[int] = set()

    @classmethod
    def compute_hash(cls, token_ids, prefix=-1):
        h = xxhash.xxh64()
        if prefix != -1: h.update(prefix.to_bytes(8, "little"))
        h.update(np.array(token_ids).tobytes())
        return h.intdigest()

    def _allocate_block(self, block_id):
        block = self.blocks[block_id]; assert block.ref_count == 0; block.reset()
        self.free_block_ids.remove(block_id); self.used_block_ids.add(block_id)
        return block

    def _deallocate_block(self, block_id):
        assert self.blocks[block_id].ref_count == 0
        self.used_block_ids.remove(block_id); self.free_block_ids.append(block_id)

    def can_allocate(self, seq): return len(self.free_block_ids) >= seq.num_blocks

    def allocate(self, seq):
        assert not seq.block_table; h = -1; cache_miss = False
        for i in range(seq.num_blocks):
            token_ids = seq.block(i)
            h = self.compute_hash(token_ids, h) if len(token_ids) == self.block_size else -1
            block_id = self.hash_to_block_id.get(h, -1)
            if block_id == -1 or self.blocks[block_id].token_ids != token_ids:
                cache_miss = True
            if cache_miss:
                block_id = self.free_block_ids[0]; block = self._allocate_block(block_id)
            else:
                seq.num_cached_tokens += self.block_size
                if block_id in self.used_block_ids:
                    block = self.blocks[block_id]; block.ref_count += 1
                else:
                    block = self._allocate_block(block_id)
            if h != -1:
                block.update(h, token_ids); self.hash_to_block_id[h] = block_id
            seq.block_table.append(block_id)

    def deallocate(self, seq):
        for block_id in reversed(seq.block_table):
            block = self.blocks[block_id]; block.ref_count -= 1
            if block.ref_count == 0: self._deallocate_block(block_id)
        seq.num_cached_tokens = 0; seq.block_table.clear()

    def can_append(self, seq): return len(self.free_block_ids) >= (len(seq) % self.block_size == 1)

    def may_append(self, seq):
        block_table = seq.block_table; last_block = self.blocks[block_table[-1]]
        if len(seq) % self.block_size == 1:
            assert last_block.hash != -1
            block_id = self.free_block_ids[0]; self._allocate_block(block_id); block_table.append(block_id)
        elif len(seq) % self.block_size == 0:
            assert last_block.hash == -1
            token_ids = seq.block(seq.num_blocks-1)
            prefix = self.blocks[block_table[-2]].hash if len(block_table) > 1 else -1
            h = self.compute_hash(token_ids, prefix)
            last_block.update(h, token_ids); self.hash_to_block_id[h] = last_block.block_id
        else:
            assert last_block.hash == -1


class Scheduler:
    def __init__(self, config: Config):
        self.max_num_seqs = config.max_num_seqs
        self.max_num_batched_tokens = config.max_num_batched_tokens
        self.eos = config.eos
        self.block_manager = BlockManager(config.num_kvcache_blocks, config.kvcache_block_size)
        self.waiting: deque[Sequence] = deque()
        self.running: deque[Sequence] = deque()

    def is_finished(self): return not self.waiting and not self.running
    def add(self, seq): self.waiting.append(seq)

    def schedule(self):
        scheduled_seqs = []; num_seqs = 0; num_batched_tokens = 0
        while self.waiting and num_seqs < self.max_num_seqs:
            seq = self.waiting[0]
            if num_batched_tokens + len(seq) > self.max_num_batched_tokens or not self.block_manager.can_allocate(seq):
                break
            num_seqs += 1; self.block_manager.allocate(seq)
            num_batched_tokens += len(seq) - seq.num_cached_tokens
            seq.status = SequenceStatus.RUNNING
            self.waiting.popleft(); self.running.append(seq); scheduled_seqs.append(seq)
        if scheduled_seqs: return scheduled_seqs, True

        while self.running and num_seqs < self.max_num_seqs:
            seq = self.running.popleft()
            while not self.block_manager.can_append(seq):
                if self.running: self.preempt(self.running.pop())
                else: self.preempt(seq); break
            else:
                num_seqs += 1; self.block_manager.may_append(seq); scheduled_seqs.append(seq)
        assert scheduled_seqs
        self.running.extendleft(reversed(scheduled_seqs))
        return scheduled_seqs, False

    def preempt(self, seq):
        seq.status = SequenceStatus.WAITING; self.block_manager.deallocate(seq); self.waiting.appendleft(seq)

    def postprocess(self, seqs, token_ids):
        for seq, token_id in zip(seqs, token_ids):
            seq.append_token(token_id)
            if (not seq.ignore_eos and token_id == self.eos) or seq.num_completion_tokens == seq.max_tokens:
                seq.status = SequenceStatus.FINISHED
                self.block_manager.deallocate(seq); self.running.remove(seq)

class ModelRunner:
    """
    JAX-first TPU runner.

    Key changes versus the original:
    - no global Python context object on the hot path
    - separate jitted prefill/decode entrypoints
    - sampling stays inside the compiled step
    - qwen3_forward is called in a scan-friendly way
    """

    def __init__(self, config: Config, rank: int, event=None, mode: str = "tpu"):
        assert mode == "tpu", "This JAX runner targets TPU only."
        self.config = config
        self.rank = rank
        self.mode = mode
        self.block_size = config.kvcache_block_size
        # static limit from max_model_len
        self.max_blocks_per_seq = (config.max_model_len + config.kvcache_block_size - 1) // config.kvcache_block_size

        self.hf_config = config.hf_config

        print("Loading weights …")
        self.weights = Qwen3Weights(config.model, self.hf_config)
        print("Weights loaded.")

        self.kv_caches: list[KVCacheLayer] = []
        self._build_compiled_fns()

        #self.warmup_model()
        self.allocate_kv_cache()
        self.warmup_model()
        print("ModelRunner ready.")

    def _build_compiled_fns(self):
        def _prefill_step(weights, kv_caches, batch: PrefillBatch, temperatures):
            flat_input_ids = batch.input_ids.reshape(-1)
            flat_positions = batch.positions.reshape(-1)
            hidden, new_kv = qwen3_forward(
                weights=weights,
                input_ids=flat_input_ids,
                positions=flat_positions,
                kv_caches=kv_caches,
                batch=batch,
                is_prefill=True,
            )
            logits = compute_logits(hidden, weights, batch, is_prefill=True)
            token_ids = jax_sampler(logits, temperatures)
            return token_ids, new_kv

        def _decode_step(weights, kv_caches, batch: DecodeBatch, temperatures):
            hidden, new_kv = qwen3_forward(
                weights=weights,
                input_ids=batch.input_ids,
                positions=batch.positions,
                kv_caches=kv_caches,
                batch=batch,
                is_prefill=False,
            )
            logits = compute_logits(hidden, weights, batch, is_prefill=False)
            token_ids = jax_sampler(logits, temperatures)
            return token_ids, new_kv

        self._prefill_step = jax.jit(_prefill_step, donate_argnums=(1,))
        self._decode_step = jax.jit(_decode_step, donate_argnums=(1,))

    # ── Utility ───────────────────────────────────────────────────────────
    def exit(self):
        jax.effects_barrier()

    def call(self, method_name, *args):
        method = getattr(self, method_name, None)
        return method(*args)

    # ── Warmup ────────────────────────────────────────────────────────────
    def warmup_model(self):
        cfg = self.config
        max_toks = cfg.max_num_batched_tokens
        max_len = cfg.max_model_len
        num_seqs = max(1, min(max_toks // max_len, cfg.max_num_seqs))
        seqs = [Sequence([0] * max_len) for _ in range(num_seqs)]

        #warmup_kv = self._make_zero_kv_caches(num_blocks=1)
        #warmup_kv = self._make_zero_kv_caches(cfg.num_kvcache_blocks)

        prefill_batch = self.prepare_prefill(seqs)
        temps = self.prepare_sample(seqs)
        #_, warmup_kv = self._prefill_step(self.weights, warmup_kv, prefill_batch, temps)
        #jax.tree_util.tree_leaves(warmup_kv)[0].block_until_ready()
        _, self.kv_caches = self._prefill_step(self.weights, self.kv_caches, prefill_batch, temps)
        jax.tree_util.tree_leaves(self.kv_caches)[0].block_until_ready()

        decode_seqs = []
        for _ in range(num_seqs):
            s = Sequence([0] * max_len)
            s.block_table = [0]
            s.num_cached_tokens = max_len - 1
            decode_seqs.append(s)
        decode_batch = self.prepare_decode(decode_seqs)
        temps = self.prepare_sample(decode_seqs)
        #_, warmup_kv = self._decode_step(self.weights, warmup_kv, decode_batch, temps)
        #jax.tree_util.tree_leaves(warmup_kv)[0].block_until_ready()
        _, self.kv_caches = self._decode_step(self.weights, self.kv_caches, decode_batch, temps)
        jax.tree_util.tree_leaves(self.kv_caches)[0].block_until_ready()
        print("finish warmup")

    def _make_zero_kv_caches(self, num_blocks: int) -> list[KVCacheLayer]:
        hf = self.hf_config
        head_dim = getattr(hf, "head_dim", hf.hidden_size // hf.num_attention_heads)
        return [
            KVCacheLayer(num_blocks, self.block_size, hf.num_key_value_heads, head_dim)
            for _ in range(hf.num_hidden_layers)
        ]

    # ── KV-cache allocation ───────────────────────────────────────────────
    def allocate_kv_cache(self):
        cfg = self.config
        hf = self.hf_config
        head_dim = getattr(hf, "head_dim", hf.hidden_size // hf.num_attention_heads)

        try:
            stats = jax.devices()[0].memory_stats()
            total = stats["bytes_limit"]
            used = stats["bytes_in_use"]
        except Exception:
            total = 32 * 1024**3
            used = 4 * 1024**3

        dtype_bytes = 2
        block_bytes = (
            2
            * hf.num_hidden_layers
            * self.block_size
            * hf.num_key_value_heads
            * head_dim
            * dtype_bytes
        )
        num_blocks = 168
        if num_blocks > 2000 or num_blocks < 1:
            num_blocks = 168
        cfg.num_kvcache_blocks = num_blocks

        print(f"total: {total / 1024 / 1024:.3f}MB")
        print(f"used:  {used / 1024 / 1024:.3f}MB")
        print(f"block size: {block_bytes / 1024 / 1024:.3f}MB")
        print(f"# of blocks: {num_blocks}")

        self.kv_caches = self._make_zero_kv_caches(num_blocks)

    # ── Input preparation ─────────────────────────────────────────────────
    def prepare_prefill(self, seqs: list) -> PrefillBatch:
        batch_size = len(seqs)
        max_seq_len = max(len(s) - s.num_cached_tokens for s in seqs)
        import math
        bucket_len = int(2 ** math.ceil(math.log2(max(max_seq_len, 128))))

        input_ids = np.zeros((batch_size, bucket_len), dtype=np.int32)
        positions = np.zeros((batch_size, bucket_len), dtype=np.int32)
        padding_mask = np.zeros((batch_size, bucket_len), dtype=np.bool_)
        slot_mapping = np.zeros((batch_size, bucket_len), dtype=np.int32)
        cu_seqlens_q = np.zeros((batch_size + 1,), dtype=np.int32)

        for i, seq in enumerate(seqs):
            actual_len = len(seq) - seq.num_cached_tokens
            tokens = seq[seq.num_cached_tokens:]
            input_ids[i, :actual_len] = tokens
            positions[i, :actual_len] = np.arange(seq.num_cached_tokens, len(seq), dtype=np.int32)
            padding_mask[i, :actual_len] = True

            write_slots = []
            if seq.block_table:
                for j in range(seq.num_cached_blocks, seq.num_blocks):
                    start = seq.block_table[j] * self.block_size
                    end = (
                        start + self.block_size
                        if j != seq.num_blocks - 1
                        else start + seq.last_block_num_tokens
                    )
                    write_slots.extend(range(start, end))
            if write_slots:
                slot_mapping[i, :actual_len] = np.asarray(write_slots[:actual_len], dtype=np.int32)

            # Index of last real token in the *padded* flat [B*S] sequence.
            # Matches the reference: cu_seqlens_q[i+1] = i * bucket_len + actual_len
            cu_seqlens_q[i + 1] = i * bucket_len + actual_len

        return PrefillBatch(
            input_ids=jnp.asarray(input_ids),
            positions=jnp.asarray(positions),
            padding_mask=jnp.asarray(padding_mask),
            slot_mapping=jnp.asarray(slot_mapping),
            cu_seqlens_q=jnp.asarray(cu_seqlens_q),
        )

    def prepare_decode(self, seqs: list) -> DecodeBatch:
        input_ids = np.asarray([seq.last_token for seq in seqs], dtype=np.int32)
        positions = np.asarray([len(seq) - 1 for seq in seqs], dtype=np.int32)
        context_lens = np.asarray([len(seq) for seq in seqs], dtype=np.int32)
        slot_mapping = np.asarray(
            [
                seq.block_table[-1] * self.block_size + seq.last_block_num_tokens - 1
                for seq in seqs
            ],
            dtype=np.int32,
        )

        #max_blocks = max(len(seq.block_table) for seq in seqs)
        max_blocks = self.max_blocks_per_seq # static
        block_tables = np.full((len(seqs), max_blocks), -1, dtype=np.int32)
        for i, seq in enumerate(seqs):
            block_tables[i, : len(seq.block_table)] = np.asarray(seq.block_table, dtype=np.int32)

        return DecodeBatch(
            input_ids=jnp.asarray(input_ids),
            positions=jnp.asarray(positions),
            slot_mapping=jnp.asarray(slot_mapping),
            context_lens=jnp.asarray(context_lens),
            block_tables=jnp.asarray(block_tables),
        )

    def prepare_sample(self, seqs: list):
        return jnp.asarray([s.temperature for s in seqs], dtype=jnp.float32)

    # ── Core execution ────────────────────────────────────────────────────
    def _run_internal(self, seqs, is_prefill: bool, kv_caches: list[KVCacheLayer]):
        temperatures = self.prepare_sample(seqs)
        if is_prefill:
            batch = self.prepare_prefill(seqs)
            token_ids, new_kv = self._prefill_step(self.weights, kv_caches, batch, temperatures)
        else:
            batch = self.prepare_decode(seqs)
            token_ids, new_kv = self._decode_step(self.weights, kv_caches, batch, temperatures)
        return token_ids, new_kv

    def run(self, seqs: list, is_prefill: bool) -> list[int]:
        token_ids, new_kv = self._run_internal(seqs, is_prefill, self.kv_caches)
        self.kv_caches = new_kv
        token_ids = np.asarray(token_ids).tolist()
        return token_ids


# !!!IMPORTANT!!!!
# THOSE SHOULD NOT BE MODIFIED
# llm_engine.py

class LLMEngine:

    def __init__(self, model, mode="tpu", **kwargs):
        config_fields = {field.name for field in fields(Config)}
        config_kwargs = {k: v for k, v in kwargs.items() if k in config_fields}
        config = Config(model, **config_kwargs)
        # JAX single-device: no multiprocessing needed for tp_size=1
        self.model_runner = ModelRunner(config, rank=0, mode=mode)
        self.tokenizer    = AutoTokenizer.from_pretrained(config.model, use_fast=True)
        config.eos        = self.tokenizer.eos_token_id
        self.scheduler    = Scheduler(config)
        atexit.register(self.exit)

    def exit(self):
        self.model_runner.exit()
        del self.model_runner

    def add_request(self, prompt, sampling_params: SamplingParams):
        if isinstance(prompt, str):
            prompt = self.tokenizer.encode(prompt)
        seq = Sequence(prompt, sampling_params)
        self.scheduler.add(seq)

    def step(self):
        seqs, is_prefill = self.scheduler.schedule()
        token_ids = self.model_runner.call("run", seqs, is_prefill)
        self.scheduler.postprocess(seqs, token_ids)
        outputs = [(seq.seq_id, seq.completion_token_ids)
                   for seq in seqs if seq.is_finished]
        num_tokens = (sum(len(seq) for seq in seqs) if is_prefill else -len(seqs))
        return outputs, num_tokens

    def is_finished(self):
        return self.scheduler.is_finished()

    def generate(self, prompts, sampling_params, use_tqdm=True):
        if use_tqdm:
            pbar = tqdm(total=len(prompts), desc="Generating", dynamic_ncols=True)
        if not isinstance(sampling_params, list):
            sampling_params = [sampling_params] * len(prompts)
        for prompt, sp in zip(prompts, sampling_params):
            self.add_request(prompt, sp)
        outputs = {}

        # --- NEW: Track cumulative metrics ---
        total_prefill_tokens = 0
        total_prefill_time = 0.0
        total_decode_tokens = 0
        total_decode_time = 0.0

        while not self.is_finished():
            t = perf_counter()
            output, num_tokens = self.step()
            elapsed = perf_counter() - t

            if num_tokens > 0:
                # Prefill phase
                total_prefill_tokens += num_tokens
                total_prefill_time += elapsed
                current_prefill_tp = num_tokens / elapsed if elapsed > 0 else 0
                if use_tqdm:
                    pbar.set_postfix({"Prefill": f"{int(current_prefill_tp)}tok/s"})
            else:
                # Decode phase (num_tokens is negative)
                dec_tokens = -num_tokens
                total_decode_tokens += dec_tokens
                total_decode_time += elapsed
                current_decode_tp = dec_tokens / elapsed if elapsed > 0 else 0
                if use_tqdm:
                    pbar.set_postfix({"Decode": f"{int(current_decode_tp)}tok/s"})

            for seq_id, token_ids in output:
                outputs[seq_id] = token_ids
                if use_tqdm:
                    pbar.update(1)

        outputs = [outputs[seq_id] for seq_id in sorted(outputs.keys())]
        outputs = [{"text": self.tokenizer.decode(token_ids), "token_ids": token_ids} for token_ids in outputs]
        if use_tqdm:
            pbar.close()

        # --- NEW: Return the metrics dictionary ---
        metrics = {
            "prefill_tokens": total_prefill_tokens,
            "prefill_time": total_prefill_time,
            "decode_tokens": total_decode_tokens,
            "decode_time": total_decode_time
        }

        return outputs, metrics


# !!!IMPORTANT!!!!
# THOSE SHOULD NOT BE MODIFIED
class LLM(LLMEngine):
    pass

import os
import argparse
from random import seed, randint

def main():
    parser = argparse.ArgumentParser(description="Benchmark Nano vLLM")
    parser.add_argument("--mode", type=str, required=True, choices=["gpu", "tpu"], help="Hardware mode: 'gpu' or 'tpu'")
    parser.add_argument("--model_path", type=str, default="./Qwen3-0.6B/", help="Path to the downloaded model")
    parser.add_argument("--tp_size", type=int, default=1, help="Tensor parallel size")
    parser.add_argument("--num_seqs", type=int, default=50, help="Total number of sequences to generate")
    parser.add_argument("--batch_size", type=int, default=10, help="Number of sequences per iteration (max 50)")
    args = parser.parse_args()

    if args.batch_size > 50:
        print("Warning: batch_size exceeds the maximum limit of 50. Clamping to 50.")
        args.batch_size = 50

    seed(0)
    num_seqs = args.num_seqs
    max_input_len = 256
    max_output_len = 256

    path = os.path.expanduser(args.model_path)

    print(f"--- Initializing LLM Engine in {args.mode.upper()} mode ---")

    llm = LLM(path, mode=args.mode, enforce_eager=True, tensor_parallel_size=args.tp_size, max_model_len=4096)

    prompt_token_ids = [[randint(0, 10000) for _ in range(randint(100, max_input_len))] for _ in range(num_seqs)]
    sampling_params = [SamplingParams(temperature=0.6, ignore_eos=True, max_tokens=randint(100, max_output_len)) for _ in range(num_seqs)]

    print("Running warmup benchmark...")
    # Capture and ignore the metrics for the warmup run
    warmup_ids = [[0] * 200 for _ in range(args.batch_size)]   # snaps to bucket=256
    warmup_sp  = [SamplingParams(temperature=0.6, max_tokens=1)] * args.batch_size
    _, _ = llm.generate(warmup_ids, warmup_sp, use_tqdm=False)

    print(f"Starting benchmark with {num_seqs} total sequences, chunked into batches of {args.batch_size}...")

    # Global trackers for the final average
    overall_prefill_tokens = 0
    overall_prefill_time = 0.0
    overall_decode_tokens = 0
    overall_decode_time = 0.0

    for i in range(0, num_seqs, args.batch_size):
        batch_prompts = prompt_token_ids[i:i + args.batch_size]
        batch_params = sampling_params[i:i + args.batch_size]

        iteration = (i // args.batch_size) + 1
        print(f"\n--- Iteration {iteration} ({len(batch_prompts)} sequences) ---")

        # Unpack the tuple returned by our updated generate() method
        outputs, metrics = llm.generate(batch_prompts, batch_params, use_tqdm=True)

        # Accumulate metrics
        overall_prefill_tokens += metrics["prefill_tokens"]
        overall_prefill_time += metrics["prefill_time"]
        overall_decode_tokens += metrics["decode_tokens"]
        overall_decode_time += metrics["decode_time"]

        # Calculate iteration throughputs safely
        p_tp = metrics["prefill_tokens"] / metrics["prefill_time"] if metrics["prefill_time"] > 0 else 0
        d_tp = metrics["decode_tokens"] / metrics["decode_time"] if metrics["decode_time"] > 0 else 0

        print(f"Iteration {iteration} -> Prefill: {p_tp:.2f} tok/s | Decode: {d_tp:.2f} tok/s")

    # Calculate overall throughputs safely
    final_p_tp = overall_prefill_tokens / overall_prefill_time if overall_prefill_time > 0 else 0
    final_d_tp = overall_decode_tokens / overall_decode_time if overall_decode_time > 0 else 0

    print("\n===================================================")
    print(f"Overall Prefill Throughput: {final_p_tp:.2f} tok/s ({overall_prefill_tokens} tok / {overall_prefill_time:.2f} s)")
    print(f"Overall Decode Throughput:  {final_d_tp:.2f} tok/s ({overall_decode_tokens} tok / {overall_decode_time:.2f} s)")
    print("===================================================")

    # if hasattr(llm, 'exit'):
    #     llm.exit()

if __name__ == "__main__":
    main()

Overwriting nano_vllm_v4.py


In [ ]:
!python nano_vllm_v4.py --mode tpu --num_seqs 100 --batch_size 10

SplashAttention (GQA-capable TPU Pallas kernel) loaded
--- Initializing LLM Engine in TPU mode ---
Loading weights …
Weights loaded.
total: 16126.000MB
used:  1453.763MB
block size: 28.000MB
# of blocks: 168
finish warmup
ModelRunner ready.
Running warmup benchmark...
Starting benchmark with 100 total sequences, chunked into batches of 10...

--- Iteration 1 (10 sequences) ---
Generating: 100% 10/10 [00:29<00:00,  2.92s/it, Decode=21tok/s]
Iteration 1 -> Prefill: 15577.85 tok/s | Decode: 71.41 tok/s

--- Iteration 2 (10 sequences) ---
Generating: 100% 10/10 [00:13<00:00,  1.33s/it, Decode=21tok/s]
Iteration 2 -> Prefill: 18051.34 tok/s | Decode: 114.96 tok/s

--- Iteration 3 (10 sequences) ---
Generating: 100% 10/10 [00:13<00:00,  1.40s/it, Decode=21tok/s]
Iteration 3 -> Prefill: 16800.55 tok/s | Decode: 115.70 tok/s

--- Iteration 4 (10 sequences) ---
Generating: 100% 10/10 [00:14<00:00,  1.43s/it, Decode=21tok/s]
Iteration 4 -> Prefill: 19112.34 tok/s | Decode: 129.81 tok/s

--- Iter